# ML Evaluation Fundamentals
**Goal:** Understand precision, recall, F1 — the concepts behind RAG evaluation metrics.

We build everything from scratch first, then verify with scikit-learn.

## 1. The Problem

Imagine a spam detector. It looks at 10 emails and decides: spam or not spam.

- **Actual spam emails:** 5
- **Actual real emails:** 5

Your model predicts some correctly, some wrong. How do you measure how good it is?

That's what this notebook answers.

In [1]:
# Actual labels: 1 = spam, 0 = not spam
actual    = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]

# What our model predicted
predicted = [1, 1, 1, 0, 0, 0, 0, 1, 1, 0]

# Let's see what happened
for i, (a, p) in enumerate(zip(actual, predicted)):
    status = "CORRECT" if a == p else "WRONG"
    print(f"Email {i+1}: actual={'spam' if a==1 else 'real':4s}  predicted={'spam' if p==1 else 'real':4s}  -> {status}")

Email 1: actual=spam  predicted=spam  -> CORRECT
Email 2: actual=spam  predicted=spam  -> CORRECT
Email 3: actual=spam  predicted=spam  -> CORRECT
Email 4: actual=spam  predicted=real  -> WRONG
Email 5: actual=spam  predicted=real  -> WRONG
Email 6: actual=real  predicted=real  -> CORRECT
Email 7: actual=real  predicted=real  -> CORRECT
Email 8: actual=real  predicted=spam  -> WRONG
Email 9: actual=real  predicted=spam  -> WRONG
Email 10: actual=real  predicted=real  -> CORRECT


## 2. The Confusion Matrix

Every prediction falls into one of 4 boxes:

| | Predicted Spam | Predicted Real |
|---|---|---|
| **Actually Spam** | TP (True Positive) | FN (False Negative) |
| **Actually Real** | FP (False Positive) | TN (True Negative) |

- **TP** — model said spam, it WAS spam. Correct catch.
- **FP** — model said spam, it was NOT spam. False alarm.
- **FN** — model said real, it WAS spam. Missed it.
- **TN** — model said real, it was NOT spam. Correct ignore.

In [2]:
# Build confusion matrix from scratch — no libraries
TP = FP = FN = TN = 0

for a, p in zip(actual, predicted):
    if a == 1 and p == 1:
        TP += 1
    elif a == 0 and p == 1:
        FP += 1
    elif a == 1 and p == 0:
        FN += 1
    else:
        TN += 1

print(f"True Positives  (TP): {TP}  — spam caught correctly")
print(f"False Positives (FP): {FP}  — real email flagged as spam (false alarm)")
print(f"False Negatives (FN): {FN}  — spam missed (slipped through)")
print(f"True Negatives  (TN): {TN}  — real email correctly ignored")

True Positives  (TP): 3  — spam caught correctly
False Positives (FP): 2  — real email flagged as spam (false alarm)
False Negatives (FN): 2  — spam missed (slipped through)
True Negatives  (TN): 3  — real email correctly ignored


## 3. Precision

**Question:** Of all the emails I flagged as spam, how many were ACTUALLY spam?

```
Precision = TP / (TP + FP)
```

High precision = when I say spam, I am usually right. Low false alarms.

**RAG connection:** Faithfulness in RAGAS uses this idea. Of all the claims in the answer, how many are actually supported by the retrieved context?

In [3]:
precision = TP / (TP + FP)
print(f"Precision: {precision:.2f}")
print(f"Meaning: {precision*100:.0f}% of emails I flagged as spam were actually spam")

Precision: 0.60
Meaning: 60% of emails I flagged as spam were actually spam


## 4. Recall

**Question:** Of all the ACTUAL spam emails, how many did I catch?

```
Recall = TP / (TP + FN)
```

High recall = I caught most of the spam. Low missed detections.

**RAG connection:** Context Recall in RAGAS = recall. Of all the relevant information that exists, how much did retrieval actually fetch?

In [4]:
recall = TP / (TP + FN)
print(f"Recall: {recall:.2f}")
print(f"Meaning: I caught {recall*100:.0f}% of all actual spam emails")

Recall: 0.60
Meaning: I caught 60% of all actual spam emails


## 5. The Precision-Recall Tradeoff

Here is the tension:

- To get **high recall** (catch all spam) — flag everything as spam — precision drops (lots of false alarms)
- To get **high precision** (only flag when sure) — miss some spam — recall drops

You cannot maximize both. You choose based on the cost of each mistake.

- **Spam filter:** missing spam (FN) is costly — optimize for recall
- **Medical diagnosis:** false alarm (FP) wastes resources — optimize for precision

In [5]:
print("The tradeoff:")
print(f"  Precision: {precision:.2f} — how trustworthy my 'spam' calls are")
print(f"  Recall:    {recall:.2f} — how much spam I actually caught")
print()
print(f"  FP (false alarms):  {FP} real emails wrongly flagged")
print(f"  FN (missed spam):   {FN} spam emails that slipped through")

The tradeoff:
  Precision: 0.60 — how trustworthy my 'spam' calls are
  Recall:    0.60 — how much spam I actually caught

  FP (false alarms):  2 real emails wrongly flagged
  FN (missed spam):   2 spam emails that slipped through


In [10]:
# One sentence to remember the 4 boxes:

# ▎ "True/False = was I right? Positive/Negative = what did I predict?"

# I predicted SPAM (Positive):
#   - I was RIGHT  → True Positive  (TP)
#   - I was WRONG  → False Positive (FP)

# I predicted REAL (Negative):
#   - I was RIGHT  → True Negative  (TN)
#   - I was WRONG  → False Negative (FN)

# Then precision and recall follow from one question each:

# Precision → "Of everything I called SPAM, how many were actually spam?"
#              TP / (TP + FP)
#              Only involves the SPAM column of my predictions.

# Recall    → "Of all ACTUAL SPAM, how many did I catch?"
#              TP / (TP + FN)
#              Only involves the ACTUAL SPAM row.

# ---
# Add this cell and run it — it prints the confusion matrix with labels so you can see it visually:

import numpy as np

actual    = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
predicted = [1, 1, 1, 0, 0, 0, 0, 1, 1, 0]

TP = FP = FN = TN = 0
for a, p in zip(actual, predicted):
    if   a==1 and p==1: TP += 1
    elif a==0 and p==1: FP += 1
    elif a==1 and p==0: FN += 1
    else:                TN += 1

print("           Predicted SPAM  Predicted REAL")
print(f"Actual SPAM      TP={TP}            FN={FN}    <- recall uses this row")
print(f"Actual REAL      FP={FP}            TN={TN}")
print()
print("Precision uses the SPAM column: TP / (TP + FP)")
print("Recall    uses the SPAM row:    TP / (TP + FN)")
print()
print(f"Precision = {TP} / ({TP} + {FP}) = {TP/(TP+FP):.2f}")
print(f"Recall    = {TP} / ({TP} + {FN}) = {TP/(TP+FN):.2f}")

# Run it. The visual layout should make it click.

           Predicted SPAM  Predicted REAL
Actual SPAM      TP=3            FN=2    <- recall uses this row
Actual REAL      FP=2            TN=3

Precision uses the SPAM column: TP / (TP + FP)
Recall    uses the SPAM row:    TP / (TP + FN)

Precision = 3 / (3 + 2) = 0.60
Recall    = 3 / (3 + 2) = 0.60


## 6. F1 Score

**Question:** Can I get one number that balances both?

```
F1 = 2 * (Precision * Recall) / (Precision + Recall)
```

This is the **harmonic mean** of precision and recall. It punishes extreme imbalance.

Example: precision=1.0, recall=0.1 gives F1=0.18 — not 0.55. Low recall kills the score.

F1 = 1.0 is perfect. F1 = 0.0 is useless.

In [6]:
f1 = 2 * (precision * recall) / (precision + recall)

print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1 Score:  {f1:.2f}")
print()

# Show why harmonic mean punishes imbalance
p_extreme, r_extreme = 1.0, 0.1
f1_extreme = 2 * (p_extreme * r_extreme) / (p_extreme + r_extreme)
print(f"Extreme case: precision={p_extreme}, recall={r_extreme}")
print(f"Arithmetic mean would give: {(p_extreme + r_extreme)/2:.2f}  (misleadingly high)")
print(f"F1 (harmonic mean) gives:   {f1_extreme:.2f}  (honest — punishes low recall)")

Precision: 0.60
Recall:    0.60
F1 Score:  0.60

Extreme case: precision=1.0, recall=0.1
Arithmetic mean would give: 0.55  (misleadingly high)
F1 (harmonic mean) gives:   0.18  (honest — punishes low recall)


## 7. Verify with scikit-learn

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

print("scikit-learn results (should match our manual calc):")
print(f"  Precision: {precision_score(actual, predicted):.2f}")
print(f"  Recall:    {recall_score(actual, predicted):.2f}")
print(f"  F1:        {f1_score(actual, predicted):.2f}")
print()
print("Confusion matrix [[TN FP] [FN TP]]:")
cm = confusion_matrix(actual, predicted)
print(cm)

scikit-learn results (should match our manual calc):
  Precision: 0.60
  Recall:    0.60
  F1:        0.60

Confusion matrix [[TN FP] [FN TP]]:
[[3 2]
 [2 3]]


## 8. How This Maps to RAG Evaluation

| Traditional ML | RAG (RAGAS) | What it asks |
|---|---|---|
| Precision | Faithfulness | Of all claims in the answer, how many are supported by context? |
| Recall | Context Recall | Of all relevant info that exists, how much did retrieval fetch? |
| F1 | Answer Relevancy | Does the answer actually address the question? |

RAGAS does not use labels like spam/not spam. Instead it uses an **LLM as judge** to score each metric automatically.

The underlying math intuition is identical to what you just built.

In [8]:
# Concrete RAG example — trace through the logic
print("=== RAG Example ===")
print()
print("Question:  'What is the refund policy?'")
print()
print("Context:   'Refunds within 30 days. No refunds on digital goods.'")
print()
print("Answer:    'Refunds within 30 days. No refunds on digital goods. Free shipping on all orders.'")
print()
print("--- Faithfulness (precision) ---")
print("  Claims in answer: 3")
print("  Supported by context: 2")
print("  Hallucinated: 1 ('Free shipping' -- not in context)")
print(f"  Faithfulness = 2/3 = {2/3:.2f}")
print()
print("--- Context Recall (recall) ---")
print("  Ground truth facts: 2")
print("  Fetched by retrieval: 2")
print(f"  Context Recall = 2/2 = {2/2:.2f}")
print()
print("The hallucination is caught by Faithfulness dropping below 1.0.")
print("This is exactly what RAGAS measures automatically using an LLM as judge.")

=== RAG Example ===

Question:  'What is the refund policy?'

Context:   'Refunds within 30 days. No refunds on digital goods.'

Answer:    'Refunds within 30 days. No refunds on digital goods. Free shipping on all orders.'

--- Faithfulness (precision) ---
  Claims in answer: 3
  Supported by context: 2
  Hallucinated: 1 ('Free shipping' -- not in context)
  Faithfulness = 2/3 = 0.67

--- Context Recall (recall) ---
  Ground truth facts: 2
  Fetched by retrieval: 2
  Context Recall = 2/2 = 1.00

The hallucination is caught by Faithfulness dropping below 1.0.
This is exactly what RAGAS measures automatically using an LLM as judge.


In [9]:
spam_probabilities = [0.95, 0.87, 0.76, 0.45, 0.38, 0.12, 0.08, 0.72, 0.65, 0.05]
actual             = [1,    1,    1,    1,    1,    0,    0,    0,    0,    0   ]

def evaluate(probs, labels, threshold):
    predicted = [1 if p >= threshold else 0 for p in probs]
    TP = FP = FN = TN = 0
    for a, p in zip(labels, predicted):
        if   a==1 and p==1: TP += 1
        elif a==0 and p==1: FP += 1
        elif a==1 and p==0: FN += 1
        else:                TN += 1
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1        = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 44)
for t in [0.3, 0.5, 0.6, 0.7, 0.8, 0.9]:
    p, r, f = evaluate(spam_probabilities, actual, t)
    print(f"{t:>10.1f} {p:>10.2f} {r:>10.2f} {f:>10.2f}")

# Run it and read the table. You'll see: lower threshold → recall goes up, precision goes down. Higher threshold → opposite. That's the tradeoff in action.

 Threshold  Precision     Recall         F1
--------------------------------------------
       0.3       0.71       1.00       0.83
       0.5       0.60       0.60       0.60
       0.6       0.60       0.60       0.60
       0.7       0.75       0.60       0.67
       0.8       1.00       0.40       0.57
       0.9       1.00       0.20       0.33
